# 5. Annotate prepared windows with local Qwen

Load the interim prepared-window manifests, run the local MLX Qwen3-VL model, and write event annotations to `data/processed/<video-name>/window_event_annotations.json`.

This notebook must run natively on Apple Silicon with `mlx-vlm` installed; MLX/Metal is not available in the Airflow Docker container.

In [ ]:
import json
from pathlib import Path

import mlx.core as mx
from videometa import LocalQwenEventAnnotator, PreparedWindowInput

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_ID = "mlx-community/Qwen3-VL-4B-Instruct-4bit"

mx.set_cache_limit(2 * 1024**3)
annotator = LocalQwenEventAnnotator(
    model_id=MODEL_ID,
    video_fps=2.0,
    max_tokens=600,
    prompt_token_budget=6000,
    feature_frames=8,
)

manifest_paths = sorted(INTERIM_DIR.glob("*/prepared_windows.json"))
if not manifest_paths:
    raise FileNotFoundError(f"No prepared window manifests found in {INTERIM_DIR}")

for manifest_path in manifest_paths:
    video_name = manifest_path.parent.name
    items = json.loads(manifest_path.read_text(encoding="utf-8"))
    output_dir = PROCESSED_DIR / video_name
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "window_event_annotations.json"

    results = []
    for item in items:
        prepared = PreparedWindowInput(
            start_seconds=item["start_seconds"],
            end_seconds=item["end_seconds"],
            object_features=tuple(item["object_features"]),
            frame_features=tuple(item["frame_features"]),
            image_messages=tuple(item.get("image_messages", [])),
            artifact_directory=item.get("artifact_directory"),
            annotated_video_path=item.get("annotated_video_path"),
        )
        try:
            events = annotator.annotate(prepared)
            results.append(
                {
                    "start_seconds": prepared.start_seconds,
                    "end_seconds": prepared.end_seconds,
                    "annotated_video_path": prepared.annotated_video_path,
                    "events": events,
                }
            )
        except Exception as error:
            results.append(
                {
                    "start_seconds": prepared.start_seconds,
                    "end_seconds": prepared.end_seconds,
                    "annotated_video_path": prepared.annotated_video_path,
                    "events": [],
                    "error": str(error),
                }
            )
        output_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    print(f"Wrote {len(results)} window annotations to {output_path}")